# Fine-tuning Qwen2-0.5B pour l'autocompletion — domaine detection incendie (ISO 7240-14)

Notebook **jumeau** de `finetune_gpt2_fire_detection.ipynb`, avec un modele plus recent.

Meme pipeline exactement (corpus nettoye, split 70/15/15, early stopping,
regularisation, perplexite + Top-1/Top-3) -> les chiffres sont **directement comparables**
a ceux de GPT-2.

**Pourquoi Qwen2-0.5B ?** Taille proche de GPT-2 (~0.5B vs 124M) donc comparaison
equitable, mais modele de 2024 entraine sur de bien meilleures donnees.
Objectif : voir si un petit modele plus recent fait mieux que GPT-2 sur le meme protocole.


## 0. Verifier le GPU

In [2]:
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "PAS DE GPU (Execution > Modifier le type d'execution > GPU)")

GPU : Tesla T4


In [3]:
!pip install -q transformers datasets accelerate

## 1. Monter Google Drive

On reutilise le meme projet Drive et le meme corpus que pour GPT-2.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/autocomplete'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
print("Projet :", PROJECT_DIR)

Mounted at /content/drive
Projet : /content/drive/MyDrive/autocomplete


## 2. Charger le corpus nettoye

Le meme `scenarios_fire_detection_clean.txt`. Adapte le chemin si besoin
(racine du Drive ou dossier data).

In [5]:
CLEAN_CORPUS = '/content/drive/MyDrive/scenarios_fire_detection_clean.txt'  # adapte si besoin

if not os.path.exists(CLEAN_CORPUS):
    print("Corpus absent -> upload manuel")
    from google.colab import files
    up = files.upload()
    fname = list(up.keys())[0]
    with open(CLEAN_CORPUS, 'wb') as f:
        f.write(up[fname])

with open(CLEAN_CORPUS, encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Corpus charge : {len(lines)} phrases")
print("Exemple :", lines[0])

Corpus charge : 301 phrases
Exemple : Fire detection and alarm systems — Part 14: design, installation, commissioning and service of fire detection and fire alarm systems in and around buildings.


## 3. Split 3-way (70 / 15 / 15) — **meme seed que GPT-2**

On garde `seed=42` : le set de **test est identique** a celui utilise pour GPT-2.
C'est indispensable pour que la comparaison des chiffres soit valable.

In [6]:
import random
random.seed(42)
shuffled = lines[:]
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_lines = shuffled[:n_train]
val_lines   = shuffled[n_train:n_train + n_val]
test_lines  = shuffled[n_train + n_val:]

print(f"Train      : {len(train_lines)}")
print(f"Validation : {len(val_lines)}")
print(f"Test       : {len(test_lines)}")

Train      : 210
Validation : 45
Test       : 46


## 4. Charger Qwen2-0.5B

Differences avec GPT-2 :
- modele : `Qwen/Qwen2-0.5B`
- le dropout se regle via les arguments du modele (pas la config GPT2)
- le `pad_token` existe deja le plus souvent ; sinon on le definit

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = 'Qwen/Qwen2-0.5B'
MODEL_DIR  = f'{PROJECT_DIR}/models/qwen2_fire_detection'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
print("Qwen2-0.5B charge. Vocab :", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen2-0.5B charge. Vocab : 151646


In [8]:
from datasets import Dataset

def make_ds(lines):
    return Dataset.from_dict({'text': lines})

def tokenize_fn(ex):
    out = tokenizer(ex['text'], truncation=True, padding='max_length', max_length=64)
    out['labels'] = out['input_ids'].copy()
    return out

train_ds = make_ds(train_lines).map(tokenize_fn, batched=True, remove_columns=['text'])
val_ds   = make_ds(val_lines).map(tokenize_fn,   batched=True, remove_columns=['text'])
print("Tokenisation OK")

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenisation OK


## 5. Fine-tuning avec early stopping

**Memes reglages que GPT-2** (epochs plafond 10 + early stopping, weight decay 0.05,
petit batch, meilleur modele garde) pour une comparaison equitable.
Seul `learning_rate` est legerement abaisse (2e-5), plus sur pour Qwen.

In [13]:
from transformers import (DataCollatorForLanguageModeling, Trainer,
                          TrainingArguments, EarlyStoppingCallback)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=20,
    weight_decay=0.05,
    lr_scheduler_type='cosine',
    bf16=torch.cuda.is_available(),
    fp16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print("Fine-tuning termine. Modele sauve :", MODEL_DIR)

Epoch,Training Loss,Validation Loss
1,3.294353,2.929590
2,2.125314,2.722276
3,1.479010,2.784309
4,0.938085,3.003423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning termine. Modele sauve : /content/drive/MyDrive/autocomplete/models/qwen2_fire_detection


## 6. Perplexite sur le test (base vs fine-tune)

Meme calcul que pour GPT-2. Plus c'est bas, mieux c'est.

In [15]:
import math

def perplexity(m, lines):
    m.eval()
    device = next(m.parameters()).device
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for line in lines:
            enc = tokenizer(line, return_tensors='pt', truncation=True, max_length=64).to(device)
            out = m(**enc, labels=enc['input_ids'])
            n_tok = enc['input_ids'].size(1)
            total_loss   += out.loss.item() * n_tok
            total_tokens += n_tok
    return math.exp(total_loss / total_tokens)

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
if torch.cuda.is_available(): base = base.cuda()

ppl_base = perplexity(base, test_lines)
ppl_ft   = perplexity(model, test_lines)
print(f"Perplexite Qwen2 de base   : {ppl_base:.2f}")
print(f"Perplexite Qwen2 fine-tune : {ppl_ft:.2f}")
print(f"Amelioration               : {(ppl_base - ppl_ft)/ppl_base*100:.1f} %")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Perplexite Qwen2 de base   : 45.11
Perplexite Qwen2 fine-tune : 13.36
Amelioration               : 70.4 %


## 7. Top-1 / Top-3 sur le test (logits directs)

Meme methode fiabilisee que pour GPT-2 : on lit les logits du prochain token
et on garde les k meilleurs mots commencant par le prefixe (2 lettres).

In [16]:
import re

def tokenize_words(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

vocab_size = len(tokenizer)
tok_first_alpha = []
for tid in range(vocab_size):
    s = tokenizer.decode([tid]).strip().lower()
    tok_first_alpha.append(s if re.match(r"[a-z]", s) else "")

def predict_topk(m, context_words, prefix, k=3):
    device = next(m.parameters()).device
    sentence = " ".join(context_words).strip() or tokenizer.eos_token
    enc = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = m(**enc).logits[0, -1, :]
    order = torch.argsort(logits, descending=True).tolist()
    preds, seen = [], set()
    for tid in order:
        w = tok_first_alpha[tid]
        if not w: continue
        w = re.sub(r"[^a-z]", "", w)
        if not w or w in seen: continue
        if prefix and not w.startswith(prefix): continue
        preds.append(w); seen.add(w)
        if len(preds) >= k: break
    return preds

def eval_topk(m, lines, prefix_len=2):
    top1, top3, total = 0, 0, 0
    for line in lines:
        words = tokenize_words(line)
        for i in range(1, len(words)):
            target = words[i]
            if len(target) <= prefix_len: continue
            preds = predict_topk(m, words[:i], target[:prefix_len], k=3)
            if preds and preds[0] == target: top1 += 1
            if target in preds:              top3 += 1
            total += 1
    return {"Top-1": round(top1/total*100,1) if total else 0,
            "Top-3": round(top3/total*100,1) if total else 0,
            "tests": total}

print("Evaluation Top-k sur le test...")
res_ft = eval_topk(model, test_lines)
print("Qwen2 fine-tune :", res_ft)

Evaluation Top-k sur le test...
Qwen2 fine-tune : {'Top-1': 77.3, 'Top-3': 86.9, 'tests': 671}


In [17]:
# --- Comparaison : Qwen2 de base (non fine-tune) sur le MEME test ---
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
if torch.cuda.is_available():
    base = base.cuda()

print("Evaluation Top-k du Qwen2 de base...")
res_base = eval_topk(base, test_lines)

print()
print("=== Comparaison Top-k (meme test) ===")
print("Qwen2 de base   :", res_base)
print("Qwen2 fine-tune :", res_ft)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluation Top-k du Qwen2 de base...

=== Comparaison Top-k (meme test) ===
Qwen2 de base   : {'Top-1': 62.6, 'Top-3': 77.6, 'tests': 671}
Qwen2 fine-tune : {'Top-1': 77.3, 'Top-3': 86.9, 'tests': 671}


In [20]:
import re, torch

def next_words(sentence, k=5):
    """Donne les k mots les plus probables apres `sentence`, avec le modele Qwen2 fine-tune."""
    device = next(model.parameters()).device
    enc = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]
    order = torch.argsort(logits, descending=True).tolist()
    preds, seen = [], set()
    for tid in order:
        w = tokenizer.decode([tid]).strip().lower()
        w = re.sub(r"[^a-z]", "", w)
        if w and w not in seen:
            preds.append(w); seen.add(w)
        if len(preds) >= k:
            break
    return preds

phrase = ""
while True:
    mot = input("mot suivant (vide pour arreter) : ")
    if not mot.strip():
        break
    phrase = (phrase + " " + mot).strip()
    print(f"  phrase : {phrase}")
    print(f"  suggestions -> {next_words(phrase)}")

mot suivant (vide pour arreter) : the
  phrase : the
  suggestions -> ['the', 'log', 'single', 'installation', 'manual']
mot suivant (vide pour arreter) : fire
  phrase : the fire
  suggestions -> ['alarm', 'detection', 'control', 'protection', 'detector']
mot suivant (vide pour arreter) : detection
  phrase : the fire detection
  suggestions -> ['and', 'system', 'systems', 'control', 'equipment']
mot suivant (vide pour arreter) : system
  phrase : the fire detection system
  suggestions -> ['shall', 'is', 'may', 'should', 'and']
mot suivant (vide pour arreter) : 
